# Baseline audit: Polish hate speech detection

This notebook evaluates a Polish hate-speech classifier on a binary task (harmful vs non-harmful), checks performance by functional category, and performs a simple gender-associated name counterfactual check.

## Models
- **Primary**: `ptaszynski/bert-base-polish-cyberbullying`
- **Secondary (optional)**: set `SECOND_MODEL_NAME` to another Polish-oriented transformer classifier to compare results.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.inference import DEFAULT_MODEL_NAME, predict_binary
from src.metrics import binary_metrics, classification_summary, confusion_matrix_df
from src.counterfactuals import gender_counterfactual_pairs


In [ ]:
data_path = ROOT / 'data' / 'test.csv'
df = pd.read_csv(data_path)
df.head()


In [ ]:
PRIMARY_MODEL_NAME = DEFAULT_MODEL_NAME
SECOND_MODEL_NAME = None  # e.g., 'YOUR_SECOND_POLISH_MODEL'


In [ ]:
pred_df = predict_binary(df['text'].tolist(), model_name=PRIMARY_MODEL_NAME)
eval_df = df.join(pred_df[['model_label', 'score', 'harmful_pred']])
eval_df.head()


In [ ]:
print(binary_metrics(eval_df['label'], eval_df['harmful_pred']))
print(classification_summary(eval_df['label'], eval_df['harmful_pred']))
confusion_matrix_df(eval_df['label'], eval_df['harmful_pred'])


In [ ]:
category_scores = (
    eval_df.groupby('category')
    .apply(lambda g: pd.Series(binary_metrics(g['label'], g['harmful_pred'])))
    .sort_values('f1', ascending=False)
)
category_scores


In [ ]:
counterfactual_rows = []
for text in df['text']:
    for original, swapped in gender_counterfactual_pairs(text):
        counterfactual_rows.append({'original': original, 'swapped': swapped})

cf_df = pd.DataFrame(counterfactual_rows)
cf_df.head()


In [ ]:
if not cf_df.empty:
    orig_pred = predict_binary(cf_df['original'].tolist(), model_name=PRIMARY_MODEL_NAME)
    swap_pred = predict_binary(cf_df['swapped'].tolist(), model_name=PRIMARY_MODEL_NAME)
    sensitivity = (orig_pred['harmful_pred'] != swap_pred['harmful_pred']).mean()
    print(f'Prediction flip rate under name swap: {sensitivity:.3f}')
else:
    print('No name-based counterfactual pairs found in the dataset.')


In [ ]:
results_dir = ROOT / 'results'
results_dir.mkdir(exist_ok=True)
eval_df.to_csv(results_dir / 'baseline_primary_predictions.csv', index=False)
print('Saved predictions to results/baseline_primary_predictions.csv')
